# Part 1: Data Profiling & Data Quality Check

Working with French road accident data (BAAC), the 2024 extract. Source: ONISR / data.gouv.fr, and the codebook is ONISR's "Description des bases de donnees
annuelles des accidents corporels de la circulation routiere."

I'm loading everything in as raw strings first. Why? Because pandas' default NaN detection would just swallow the sentinel codes this dataset uses instead of actual nulls (`-1`, blank cells, `"N/A"`) — and catching that is basically the whole point of section B below.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

BRONZE = 'data/bronze'


In [2]:
def load_raw(name):
    return pd.read_csv(f'{BRONZE}/{name}', sep=';', encoding='utf-8', dtype=str, keep_default_na=False)

accidents = load_raw('caract-2024.csv')   # caracteristiques
lieux     = load_raw('lieux-2024.csv')
vehicules = load_raw('vehicules-2024.csv')
usagers   = load_raw('usagers-2024.csv')

for name, df in [('accidents', accidents), ('lieux', lieux), ('vehicules', vehicules), ('usagers', usagers)]:
    print(f'{name:12s} shape={df.shape}')


accidents    shape=(54402, 15)
lieux        shape=(70248, 18)
vehicules    shape=(92678, 11)
usagers      shape=(125187, 16)


Just to give a sense of scale here: 54,402 accidents, 92,678 vehicles, and 125,187 people involved. That's a lot of rows to keep straight.

## A. Dataset structure

Every field comes in as a plain CSV string. Honestly, picking the dtype is a modelling choice here, not something pandas can just figure out on its own — this dataset leans on sentinel codes instead of real nulls (more on that in §B). The tables below list the target dtype and what each column actually means, based on the ONISR codebook.


In [3]:
for name, df in [('accidents', accidents), ('lieux', lieux), ('vehicules', vehicules), ('usagers', usagers)]:
    print(f'--- {name} ({df.shape[1]} cols, {df.shape[0]} rows) ---')
    print(list(df.columns))


--- accidents (15 cols, 54402 rows) ---
['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg', 'int', 'atm', 'col', 'adr', 'lat', 'long']
--- lieux (18 cols, 70248 rows) ---
['Num_Acc', 'catr', 'voie', 'v1', 'v2', 'circ', 'nbv', 'vosp', 'prof', 'pr', 'pr1', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ', 'vma']
--- vehicules (11 cols, 92678 rows) ---
['Num_Acc', 'id_vehicule', 'num_veh', 'senc', 'catv', 'obs', 'obsm', 'choc', 'manv', 'motor', 'occutc']
--- usagers (16 cols, 125187 rows) ---
['Num_Acc', 'id_usager', 'id_vehicule', 'num_veh', 'place', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'etatp']


### `caracteristiques` (meant to be one row per accident — see §C.3 for why that's not quite true)

| Column | Target dtype | Meaning |
|---|---|---|
| `Num_Acc` | string (PK) | Accident identifier, links all 4 tables |
| `jour`, `mois`, `an` | int, combine into `date` | Day / month / year |
| `hrmn` | `"HH:MM"` → time | Time of accident. 100% well-formed in this extract |
| `lum` | categorical (1-5) | Light: 1 daylight, 2 dawn/dusk, 3 night no lighting, 4 night lighting off, 5 night lighting on |
| `dep` | categorical string | INSEE department code. Corsica uses `2A`/`2B`, not `20` |
| `com` | categorical string | INSEE commune code (`dep` + 3 digits) |
| `agg` | categorical (1-2) | 1 outside built-up area, 2 inside |
| `int` | categorical (1-9) | Intersection type: 1 none, 2 X, 3 T, 4 Y, 5 >4-branch, 6 roundabout, 7 square, 8 level crossing, 9 other |
| `atm` | categorical (-1, 1-9) | Weather: 1 normal, 2 light rain, 3 heavy rain, 4 snow/hail, 5 fog, 6 wind, 7 dazzling, 8 overcast, 9 other |
| `col` | categorical (-1, 1-7) | Collision: 1 head-on, 2 rear-end, 3 side, 4 chain, 5 multiple, 6 other, 7 none |
| `adr` | free text | Postal address, documented as mainly for in-agglomeration accidents (§B) |
| `lat`, `long` | float, comma decimal | WGS84 coordinates, e.g. `"47,56277000"` |

I kept `dep` as a string and never cast it to int — Corsica's `2A`/`2B` codes and the leading zeros on
metropolitan codes would just get destroyed by an int cast.

### `lieux` (documented as "the accident's primary location," though 29% of accidents actually have more than one row — see §C.3)

| Column | Target dtype | Meaning |
|---|---|---|
| `Num_Acc` | string (FK) | Links to `caracteristiques` |
| `catr` | categorical (1,2,3,4,5,6,7,9) | Road category: motorway, national, departmental, municipal, off public network, parking, urban metropolitan, other |
| `voie` | string | Road number |
| `v1` | int | Numeric road suffix ("2 bis"), sentinel `-1` |
| `v2` | string | Alphanumeric suffix, sentinel literal `"N/A"` |
| `circ` | categorical (-1, 1-4) | 1-way, 2-way, separated carriageways, variable-assignment lanes |
| `nbv` | int | Total traffic lanes |
| `vosp` | categorical (-1,0,1,2,3) | Dedicated lane: none, cycle path, cycle lane, reserved lane |
| `prof` | categorical (-1,1-4) | Flat, slope, hilltop, hill bottom |
| `pr`, `pr1` | int | Reference milestone / distance to it. `-1` = not applicable |
| `plan` | categorical (-1,1-4) | Straight, left curve, right curve, S-curve |
| `lartpc` | float (m) | Central reservation width, only meaningful if one exists |
| `larrout` | float (m) | Carriageway width |
| `surf` | categorical (-1,1-9) | Normal, wet, puddles, flooded, snow, mud, ice, oil, other |
| `infra` | categorical (-1,0-9) | Tunnel, bridge, interchange, railway, junction, pedestrian zone, toll, construction, other |
| `situ` | categorical (-1,0-8) | Carriageway, hard shoulder, verge, pavement, cycle path, other lane, other |
| `vma` | int (km/h) | Speed limit in force |

### `vehicules` (many rows per accident, one per vehicle involved)

Columns here: `Num_Acc` (FK), `id_vehicule` (PK, links to `usagers`), `num_veh` (an alphanumeric label
like `A01`/`B01`...), `senc` (direction of travel), `catv` (vehicle category, roughly 30 codes,
**stored without zero-padding in the raw CSV**, so it's "7" not "07", see §C.2 for why that trips
things up), `obs` (fixed obstacle struck), `obsm` (mobile obstacle struck), `choc` (impact point), `manv`
(manoeuvre right before impact), `motor` (fuel/hybrid/electric/hydrogen/human/other), and `occutc`
(public-transport occupant count, buses only).

### `usagers` (one row per person involved)

`Num_Acc` (FK), `id_usager` (PK), `id_vehicule`/`num_veh` (FK), `place` (seating
position, where `10` means pedestrian/not applicable), `catu` (driver/passenger/pedestrian), `grav`,
`sexe`, `an_nais` (birth year), `trajet` (trip purpose), `secu1/2/3` (up to three safety
equipment slots), and `locp`/`actp`/`etatp` (pedestrian-only fields).

`grav` is really the core safety metric here: 1 = unharmed, 2 = killed, 3 = hospitalised injury, 4 = light
injury. No missing values, which I confirm below.


## B. Missing values and completeness

Here's the thing: this dataset doesn't consistently use a plain empty cell for "unknown." Depending
on the column, it might be `-1` (sometimes `" -1"` because of CSV quoting), the literal string
`N/A`, or an actual blank. If I only checked for blanks I'd badly understate how much is really missing,
e.g. `lieux.pr` is 0% blank but 38.95% `-1`.


In [4]:
def sentinel_report(df, table_name):
    n = len(df)
    rows = []
    for col in df.columns:
        s = df[col].str.strip()
        n_blank = (s == '').sum()
        n_neg1 = (s == '-1').sum()
        n_na = (s == 'N/A').sum()
        total = n_blank + n_neg1 + n_na
        if total:
            rows.append({
                'table': table_name, 'column': col,
                'blank_%': round(n_blank / n * 100, 2),
                'neg1_%': round(n_neg1 / n * 100, 2),
                'N/A_literal_%': round(n_na / n * 100, 2),
                'total_%': round(total / n * 100, 2),
            })
    return pd.DataFrame(rows).sort_values('total_%', ascending=False)

missing = pd.concat([
    sentinel_report(accidents, 'accidents'),
    sentinel_report(lieux, 'lieux'),
    sentinel_report(vehicules, 'vehicules'),
    sentinel_report(usagers, 'usagers'),
])
missing


,table,column,blank_%,neg1_%,N/A_literal_%,total_%
1,accidents,adr,0.00,0.00,4.24,4.25
0,accidents,col,0.00,0.01,0.00,0.01
10,lieux,lartpc,99.95,0.00,0.00,99.95
2,lieux,v2,0.01,0.00,91.33,91.34
11,lieux,larrout,0.00,69.25,0.00,69.25
8,lieux,pr1,0.00,39.05,0.00,39.05
7,lieux,pr,0.00,38.95,0.00,38.95
1,lieux,v1,0.00,23.16,0.00,23.16
0,lieux,voie,0.01,0.00,18.75,18.76
3,lieux,circ,0.00,6.20,0.00,6.20


Turns out most of those high percentages aren't really missingness at all. `lieux.lartpc` (central
reservation width) is blank 99.95% of the time simply because most roads don't have one — there's no
"not applicable" code for it, so it just shows up blank. Same story for `vehicules.occutc` (bus/coach
occupant count).

That said, two fields are worth checking by hand, since a `-1` there could actually be a real gap
rather than a structural non-answer.


In [ ]:
# etatp (pedestrian alone/accompanied/group) should be -1 for anyone who isn't a
# pedestrian - if that's true, the 91.8% overall rate isn't real missingness
pedestrians = usagers[usagers['catu'].str.strip() == '3']
non_pedestrians = usagers[usagers['catu'].str.strip() != '3']

print('etatp = -1, pedestrians:', (pedestrians['etatp'].str.strip() == '-1').mean())
print('etatp = -1, non-pedestrians:', (non_pedestrians['etatp'].str.strip() == '-1').mean())


etatp = -1, pedestrians: 0.00021274332517817252
etatp = -1, non-pedestrians: 0.9926070509388009


In [6]:
# sexe = -1 (1.91% of usagers) - codebook says hit-and-run drivers since 2021 get
# a placeholder record with unknown sex/age, let's check if that actually holds
sexe_missing = usagers[usagers['sexe'].str.strip() == '-1']
print('rows:', len(sexe_missing))
print(sexe_missing['grav'].value_counts())
print('also missing an_nais:', (sexe_missing['an_nais'].str.strip() == '').mean())


rows: 2395
grav
1    2395
Name: count, dtype: int64
also missing an_nais: 0.9979123173277662


Good news — confirmed on both counts. `etatp=-1` sits at 99.3% for non-pedestrians and near-zero for
pedestrians, so that's structural, not actually missing. And every `sexe=-1` row has `grav=1` (indemne,
unharmed), with 99.8% also missing `an_nais` — that matches the hit-and-run placeholder pattern almost
exactly.

`pr`/`pr1` and `larrout` are the other fields I wanted to check by road type, since one blanket
"X% missing" number would hide a lot of what's really going on.


In [7]:
lieux['pr_missing'] = lieux['pr'].str.strip() == '-1'
lieux['larrout_missing'] = lieux['larrout'].str.strip() == '-1'
print('pr missing rate by catr:')
print(lieux.groupby('catr')['pr_missing'].mean().round(3))
print()
print('larrout missing rate by catr:')
print(lieux.groupby('catr')['larrout_missing'].mean().round(3))


pr missing rate by catr:
catr
1    0.006
2    0.124
3    0.128
4    0.703
5    0.444
6    0.705
7    0.275
9    0.489
Name: pr_missing, dtype: float64

larrout missing rate by catr:
catr
1    0.345
2    0.518
3    0.434
4    0.981
5    1.000
6    0.993
7    0.634
9    0.996
Name: larrout_missing, dtype: float64


`pr` (the PR reference milestone) runs from 0.6% `-1` on motorways (`catr=1`) up to 70%
on parking areas (`catr=6`) — makes sense, since PR referencing simply isn't used off the main road
network. So most of that is legitimate absence, but on motorways it's a real gap that should probably
get filled in. `larrout` (carriageway width) doesn't have any documented "not applicable" code at all,
so its 69% overall `-1` rate is a genuine gap everywhere, not just on minor roads, low-impact, but real.

One nice surprise: `caracteristiques.lat`/`long` are 0% missing (checked in §C.1) — complete coordinate
coverage, which goes against the usual assumption people make about this dataset.

### Critical missingness

`usagers.sexe = -1` (1.91%, 2,395 rows) is the one field that could actually mislead someone if you're
not careful. Every single one of these rows has `grav=1` (unharmed), and 99.8% also have a blank
`an_nais` — which lines up exactly with the codebook's note that hit-and-run drivers added since 2021
get a placeholder record with unknown sex and age. Drop the rows and real accidents disappear from
severity/volume counts; impute a value instead and hit-and-run involvement gets silently understated.
So I keep the row and recode it to an explicit "unknown, fled" category.

Age (`an_nais` blank, 2.06%, 2,579 rows) matters a lot because it drives almost every road-safety
breakdown out there, young-driver risk, elderly-pedestrian risk. Any age-bucketed KPI would silently
drop this 2%, and it's disproportionately the same hit-and-run population from above. My approach:
derive `age` where I can, flag `age_unknown` everywhere else. Imputing a mean age would just bias the
young and old tails toward the middle, which honestly feels worse than leaving an honest gap.

`lieux.circ`/`nbv`/`vosp`/`vma` sit at around 5-6% real `-1` and feed directly into road-design
risk factors, so dropping those rows would bias any "road configuration vs. severity"
analysis toward the better-documented majority (mostly motorway/urban).

### Fixing this by category, not by one blanket rule

Structurally not-applicable fields (`lartpc`, `occutc`, `v2`, `etatp`/`locp`/`actp` for
non-pedestrians, `secu2`/`secu3` for single-equipment people): I recode the sentinel to an
explicit `"not_applicable"` category. No imputing here, that would just invent facts that aren't there.

Real "not recorded" sentinels that still carry analytical weight (`sexe`, `an_nais`, `circ`, `nbv`,
`vosp`, `vma`, `larrout`): recode to `"unknown"`, keep the row. Dropping would lose more through
denominator distortion than it would gain.

`pr`/`pr1`/`voie`: this one depends on `catr`. I treat `-1` as a real gap on motorways/national
roads, but as structural absence on parking/municipal roads. One rule across all `catr` values
would be wrong in both directions.


## C. Consistency and validity checks

### C.1 Coordinates

France's open data covers metropolitan France and the DOM-TOM territories, which sit nowhere
near each other geographically. A single bounding box would flag every Réunion or Guadeloupe accident
as an error, so the check has to be done per-territory, keyed off `dep`.


In [8]:
territory_bbox = {
    'metro': (41.0, 51.5, -5.5, 9.7),
    '971':   (15.8, 16.6, -61.9, -60.9),   # Guadeloupe
    '972':   (14.3, 15.0, -61.3, -60.7),   # Martinique
    '973':   (1.8,  6.0,  -54.7, -51.5),   # Guyane
    '974':   (-21.5, -20.7, 55.1, 55.9),   # Reunion
    '975':   (46.6, 47.2, -56.5, -56.0),   # St-Pierre-et-Miquelon
    '976':   (-13.1, -12.5, 44.9, 45.4),   # Mayotte
    '977':   (17.8, 18.0, -62.9, -62.7),   # St-Barthelemy
    '978':   (17.9, 18.2, -63.2, -62.9),   # St-Martin
    '986':   (-14.4, -13.1, -178.3, -176.0),  # Wallis-et-Futuna
    '987':   (-28.0, -7.8, -155.0, -134.5),   # Polynesie francaise
    '988':   (-22.8, -19.4, 163.4, 168.2),    # Nouvelle-Caledonie
}
dom_codes = {'971', '972', '973', '974', '975', '976', '977', '978', '986', '987', '988'}

lat = accidents['lat'].str.replace(',', '.').astype(float)
lon = accidents['long'].str.replace(',', '.').astype(float)

def coord_ok(dep, lat_val, lon_val):
    box = territory_bbox[dep if dep in dom_codes else 'metro']
    return box[0] <= lat_val <= box[1] and box[2] <= lon_val <= box[3]

valid_coord = [coord_ok(d, la, lo) for d, la, lo in zip(accidents['dep'], lat, lon)]
bad_coords = accidents.loc[~pd.Series(valid_coord, index=accidents.index)]
print(f'{len(bad_coords)} / {len(accidents)} outside their own territory bbox')
bad_coords[['Num_Acc', 'dep', 'lat', 'long']]


10 / 54402 outside their own territory bbox


,Num_Acc,dep,lat,long
2079,202400002080,976,"45,16376000","-12,78954000"
5347,202400005348,976,"45,23097000","12,75862000"
8234,202400008235,2B,"9,44460000","42,67720000"
11272,202400011273,2B,"9,44085000","42,65160000"
13465,202400013466,2B,"9,44430000","42,67720000"
18638,202400018639,976,"45,23233000","-12,78381000"
21732,202400021733,976,"45,23507000","12,78223000"
24443,202400024444,976,"45,22950000","-12,78421000"
37078,202400037079,976,"-12,46332000","45,22346000"
49893,202400049894,976,"45,22602000","-12,76237000"


Ten rows show up, all from Mayotte (`dep=976`) or Haute-Corse (`dep=2B`). Lat and long are
swapped, not randomly wrong: Mayotte's real position is roughly lat -12.8 / long 45.2,
and these rows have lat≈45.2 / long≈-12.8. That's fixable row by row, not a "drop it" case.


In [9]:
print('exact (0,0):', ((lat == 0) & (lon == 0)).sum())
print('blank lat/long:', (accidents['lat'].str.strip() == '').sum(), (accidents['long'].str.strip() == '').sum())


exact (0,0): 0
blank lat/long: 0 0


No `(0,0)` placeholders, no blanks anywhere. This 2024 extract has complete coordinate
coverage, worth pointing out since missing GPS is usually just assumed for this dataset. That
assumption actually drove one design decision worth flagging here rather than later: going in, I
expected I'd need a fallback strategy for missing coordinates. Turns out there's nothing to
fall back to in this extract, so I didn't build one, adding a fallback for a problem that
doesn't actually exist in the data would be exactly the kind of speculative code this
project is trying to avoid. What did carry over from that line of thinking is the
per-territory bounding-box check above, which ended up catching something real instead: the 10
swapped rows below.

### C.2 Categorical oddities


In [10]:
documented = {
    'lum': {'1','2','3','4','5'},
    'atm': {'-1','1','2','3','4','5','6','7','8','9'},
    'col': {'-1','1','2','3','4','5','6','7'},
    'int': {'1','2','3','4','5','6','7','8','9'},
    'agg': {'1','2'},
}
for col, allowed in documented.items():
    actual = set(accidents[col].str.strip())
    print(col, 'unexpected values:', actual - allowed)


lum unexpected values: set()
atm unexpected values: set()


col unexpected values: set()


int unexpected values: set()
agg unexpected values: set()


All five line up exactly with their documented code lists, no anomalies in
`caracteristiques`. `vehicules.catv`, though, is a different story.


In [11]:
documented_catv = {
    '00','01','02','03','07','10','13','14','15','16','17','20','21',
    '30','31','32','33','34','35','36','37','38','39','40','41','42','43',
    '50','60','80','99','-1',
}
actual_catv = set(vehicules['catv'].str.strip())
print('not in documented set:', actual_catv - documented_catv)


not in documented set: {'2', '1', '0', '7', '3'}


`{'0','1','2','3','7'}` look invalid at first glance against the codebook's two-digit codes
(`00`, `01`, ...). They're not, the CSV just stores single digits without the leading
zero. A plain string match against the codebook would wrongly flag ~35% of `catv=07`
vehicles. This is really a standardization fix (cast to int), not a data-cleaning issue.

### C.3 Speed limits (`lieux.vma`)


In [12]:
vma_numeric = lieux.loc[lieux['vma'].str.strip().str.lstrip('-').str.isdigit(), 'vma'].astype(int)
print(vma_numeric.describe())
implausible = vma_numeric[vma_numeric > 130]   # 130 km/h is France's legal ceiling
print(f'{len(implausible)} rows above 130 km/h:', sorted(implausible.unique()))


count    70248.000000
mean        53.711479
std         27.238233
min         -1.000000
25%         30.000000
50%         50.000000
75%         70.000000
max        900.000000
Name: vma, dtype: float64
29 rows above 130 km/h: [np.int64(140), np.int64(300), np.int64(301), np.int64(500), np.int64(700), np.int64(800), np.int64(900)]


`900`, `800`, `700`, `500`, `301`, `300`, `140` km/h, clearly keying errors, not real limits.
Only 29 rows out of 70,248, but each one would sit as an extreme outlier in a speed-vs-severity
chart, so worth catching.

This extract only covers 2024, so cross-year drift couldn't be checked directly, but the
codebook does document three breaks worth carrying into a multi-year pipeline: the "blessé
hospitalisé" qualification isn't comparable before/after 2018 (process change), it lost
its official statistical label in 2019, and the hit-and-run placeholders (§B) only exist
from 2021 onward. Three separate schema epochs, basically, not one continuous series.

### C.4 Duplicates


In [13]:
print('Num_Acc unique:', accidents['Num_Acc'].nunique(), '/', len(accidents))
for name, df in [('accidents', accidents), ('lieux', lieux), ('vehicules', vehicules), ('usagers', usagers)]:
    print(f'{name:12s} full-row duplicates: {df.duplicated(keep=False).sum()}')


Num_Acc unique: 54402 / 54402
accidents    full-row duplicates: 0
lieux        full-row duplicates: 0
vehicules    full-row duplicates: 0


usagers      full-row duplicates: 0


No full-row duplicates, `Num_Acc` is a clean key. Two more interesting cases below,
each with a different underlying cause.


In [ ]:
# lieux is supposedly one row per accident... let's check that
rows_per_accident = lieux.groupby('Num_Acc').size()
print(rows_per_accident.value_counts().sort_index())


1    38757
2    15463
3      164
4       17
5        1
Name: count, dtype: int64


28.8% of accidents (15,645 of 54,402) have anywhere from 2 to 5 `lieux` rows, not just one.
That directly contradicts the codebook's own "lieu principal" (singular) description. Let me check
if it lines up with intersections.


In [15]:
multi_row_ids = set(rows_per_accident[rows_per_accident > 1].index)
accidents['multi_lieu'] = accidents['Num_Acc'].isin(multi_row_ids)
accidents.groupby('multi_lieu')['int'].value_counts(normalize=True).sort_index()


multi_lieu  int
False       1      0.892458
            2      0.004593
            3      0.003328
            4      0.000464
            5      0.000052
            6      0.033671
            7      0.006063
            8      0.002503
            9      0.056867
True        2      0.390604
            3      0.404091
            4      0.065836
            5      0.016683
            6      0.090252
            7      0.013934
            8      0.002621
            9      0.015980
Name: proportion, dtype: float64

Confirmed it: 89% of single-row accidents are `int=1` (no intersection), versus only 7% of
multi-row ones. Each named road at an intersection gets its own row. Not row
duplication, a grain violation. `accidents.merge(lieux, on='Num_Acc')` would silently
multiply every joined fact for 28.8% of accidents unless the primary row is picked
first. Honestly, this is probably the most consequential finding in this whole notebook,
everything downstream that joins on `Num_Acc` has to account for it (see Part 2A below).


In [ ]:
# usagers: can two people share the exact same seat in the same vehicle?
# (excluding pedestrians, whose place is always the fixed placeholder 10)
non_ped = usagers[usagers['catu'].str.strip() != '3']
seat_conflicts = non_ped[non_ped.duplicated(['Num_Acc', 'id_vehicule', 'place'], keep=False)]
print(f'{len(seat_conflicts)} / {len(non_ped)} rows share a seat')
print(seat_conflicts['place'].value_counts().head())


1459 / 115786 rows share a seat
place
2    584
8    224
4    189
7    127
5    114
Name: count, dtype: int64


1,459 rows (1.26%), mostly `place=2` (front passenger). No vehicle has two rows
coded as driver, so it's not a system-level failure, more likely officers defaulting
to a generic seat code for multiple back-seat occupants. Different people, same key,
so I flag rather than dedup.

### C.5 Age plausibility


In [17]:
birth_year = usagers.loc[usagers['an_nais'].str.strip() != '', 'an_nais'].astype(int)
age = 2024 - birth_year
print(age.describe())
print('negative:', (age < 0).sum(), '| over 110:', (age > 110).sum())


count    122608.000000
mean         38.924589
std          19.327486
min           0.000000
25%          23.000000
50%          36.000000
75%          53.000000
max         110.000000
Name: an_nais, dtype: float64
negative: 0 | over 110: 0


Clean here, no negative ages, nothing implausibly old, at least once `an_nais` is actually present.

### C.6 What else checked out fine


In [18]:
jour = accidents['jour'].astype(int)
mois = accidents['mois'].astype(int)
an = accidents['an'].astype(int)
bad_dates = sum(1 for j, m, a in zip(jour, mois, an) if not (1 <= m <= 12 and 1 <= j <= 31))
print('an unique:', an.unique(), '| jour range:', jour.min(), jour.max(), '| mois range:', mois.min(), mois.max())
print('invalid day/month combos:', bad_dates)

bad_hrmn = (~accidents['hrmn'].str.match(r'^\d{2}:\d{2}$')).sum()
print('bad hrmn format:', bad_hrmn)

deps = set(accidents['dep'].str.strip())
metro = {f'{i:02d}' for i in range(1, 96)} - {'20'} | {'2A', '2B'}
dom = {'971','972','973','974','975','976','977','978','986','987','988'}
print('unexpected department codes:', deps - metro - dom)


an unique: [2024] | jour range: 1 31 | mois range: 1 12
invalid day/month combos: 0
bad hrmn format: 0
unexpected department codes: set()


Every `(jour, mois, an)` combination is a valid calendar date, `an` is uniformly
2024, `hrmn` is `HH:MM` for all 54,402 rows, and every department code present is a
valid INSEE code (metropolitan `01`-`95` minus the retired `20`, replaced by `2A`/`2B`,
plus the DOM-TOM codes). Nothing to fix here.


## D. Data quality summary

Ranked by likely analytical damage, not row count:

| # | Issue | Severity | Rows affected | Downstream impact |
|---|---|---|---|---|
| 1 | `lieux` grain violation | High | 15,645 accidents | Naive joins double/triple-count accident-level facts for 28.8% of accidents. The single most likely way this dataset produces a wrong headline number. |
| 2 | `sexe`/`an_nais` gaps concentrated in hit-and-run placeholders | Medium-high | ~2,400-2,600 rows | Biases demographic breakdowns, and it's non-random, concentrated in unharmed hit-and-run drivers, not missing-at-random. |
| 3 | `catv` not zero-padded vs. codebook | Medium | ~35% of `catv=07` vehicles | Silent lookup failures if matched as strings, undercounts vehicle-category breakdowns. |
| 4 | Seat-conflict duplicates | Medium | 1,459 rows | Noisy labels for per-seat analysis, doesn't affect person-count or severity totals. |
| 5 | `vma` implausible values | Low-medium | 29 rows | Small volume, each row shows up as an extreme outlier bucket in speed-vs-severity charts. |
| 6 | Coordinate swap | Low | 10 rows | Places 10 accidents in the wrong hemisphere on a map, negligible for aggregate KPIs. |
| 7 | `circ`/`nbv`/`vosp`/`vma` real non-response | Low | ~4,000 rows each | Slight bias toward better-documented (motorway/urban) locations in road-configuration analysis. |
| 8 | `larrout` non-response (69.25%) | Low | 48,646 rows | Rules it out as a reliable model input, but rarely the primary variable for core KPIs anyway. |


## Part 2: Transformations, Modeling & the Medallion Architecture

### 2A. Silver layer transformations

For each issue in Part 1, here's the fix and why, checked against the real data rather than
left as a plausible-sounding claim. Writes `data/silver/*.parquet` and
`transform_log.csv`, logging every row touched so nothing gets dropped silently.

A few standardization choices up front, before getting into any specific issue: `jour`/`mois`/
`an`/`hrmn` collapse into one `datetime` column rather than staying four string
columns, that's a parsing problem solved once instead of pushed onto every consumer.
Coordinates go from `"47,56277000"` (comma-decimal string) to a real float, comma
decimals sort and compare wrong and crash any mapping library. `catv` gets cast to int
everywhere so `7` and `07` are the same value by construction (§C.2). Categorical
sentinels (`-1`, blank, `"N/A"`) become real nulls (`pd.NA`), so `isna()` just works
instead of a manual string comparison someone forgets in one query and not another.


In [19]:
from pathlib import Path

SILVER = Path('data/silver')
SILVER.mkdir(exist_ok=True)

transform_log = []

def record(step, table, rows_affected, note):
    transform_log.append({'step': step, 'table': table, 'rows_affected': rows_affected, 'note': note})

def to_int_with_sentinel(series):
    cleaned = series.str.strip().replace({'-1': pd.NA, '': pd.NA, 'N/A': pd.NA})
    return pd.to_numeric(cleaned, errors='coerce').astype('Int64')


Re-loading from bronze fresh rather than reusing the Part 1 dataframes, some of
those picked up scratch columns (`multi_lieu`, `pr_missing`...) along the way that
don't belong in Silver.


In [ ]:
accidents = load_raw('caract-2024.csv')
n0 = len(accidents)

accidents['date'] = pd.to_datetime(
    dict(year=accidents['an'].astype(int), month=accidents['mois'].astype(int), day=accidents['jour'].astype(int))
)
accidents['heure'] = accidents['hrmn'].str.slice(0, 2).astype(int)
accidents['minute'] = accidents['hrmn'].str.slice(3, 5).astype(int)
accidents['datetime'] = accidents['date'] + pd.to_timedelta(accidents['heure'], unit='h') + pd.to_timedelta(accidents['minute'], unit='m')
record('standardize_datetime', 'caracteristiques', n0, 'combined jour/mois/an/hrmn into 1 datetime column')

accidents['lat'] = accidents['lat'].str.replace(',', '.').astype(float)
accidents['long'] = accidents['long'].str.replace(',', '.').astype(float)
record('standardize_coordinates', 'caracteristiques', n0, 'comma-decimal string to float (WGS84)')


In [21]:
# same per-territory bbox as Part 1 C.1 - re-deriving it instead of hardcoding the 10 Num_Acc values
bad_mask = ~pd.Series(
    [coord_ok(d, la, lo) for d, la, lo in zip(accidents['dep'], accidents['lat'], accidents['long'])],
    index=accidents.index,
)
swap_fixes = bad_mask & pd.Series(
    [coord_ok(d, lo, la) for d, la, lo in zip(accidents['dep'], accidents['lat'], accidents['long'])],
    index=accidents.index,
)
accidents.loc[swap_fixes, ['lat', 'long']] = accidents.loc[swap_fixes, ['long', 'lat']].values
record('fix_coordinate_swap', 'caracteristiques', int(swap_fixes.sum()), 'lat/long swapped, rows outside their territory bbox where swapping fixes it')

still_bad = bad_mask & ~swap_fixes
if still_bad.sum():
    accidents.loc[still_bad, ['lat', 'long']] = np.nan
record('null_unresolvable_coordinates', 'caracteristiques', int(still_bad.sum()), 'outside bbox and swap does not fix it, nulled rather than guessed')

for col in ['lum', 'agg', 'int', 'atm', 'col']:
    accidents[col] = to_int_with_sentinel(accidents[col])

accidents = accidents.drop(columns=['jour', 'mois', 'an', 'hrmn', 'heure', 'minute'])
accidents.to_parquet(SILVER / 'caracteristiques.parquet', index=False)
print('swapped:', int(swap_fixes.sum()), '| nulled:', int(still_bad.sum()))


swapped: 7 | nulled: 3


7 of the 10 flagged rows resolve cleanly, swapping lands the point inside the
accident's own department box. The other 3 don't: one Mayotte row has the wrong sign on
`long` plus a magnitude off by roughly 0.3 degrees, not a clean transposition. Guessing
a second correction with nothing to check it against would be worse than an honest
unknown, so those 3 are nulled for manual review rather than force-fixed.


In [ ]:
lieux_raw = load_raw('lieux-2024.csv')
n0 = len(lieux_raw)

# Judgment call: first row per Num_Acc wins as "lieu principal". This is arbitrary -
# the raw file doesn't mark which row is primary and the codebook gives no tie-break
# rule either. I picked "first row" because it's cheap to audit: a reviewer can just
# open the raw CSV and see exactly which row survived. A bridge table (accident to
# multiple roads) would be more normalized, but it just moves the same pick to query
# time later, with less visibility into why. Logging it once here instead of leaving
# every future query to decide differently.
lieux_raw['_row_order'] = range(len(lieux_raw))
lieux_silver = lieux_raw.sort_values('_row_order').groupby('Num_Acc', as_index=False).first().drop(columns='_row_order')
record('dedup_lieux_to_1_per_accident', 'lieux', n0 - len(lieux_silver), 'kept first row per Num_Acc, multi-road intersections collapse to their first-listed road')

for col in ['catr', 'circ', 'vosp', 'prof', 'plan', 'surf', 'infra', 'situ']:
    lieux_silver[col] = to_int_with_sentinel(lieux_silver[col])
lieux_silver['nbv'] = to_int_with_sentinel(lieux_silver['nbv'])
lieux_silver['vma'] = to_int_with_sentinel(lieux_silver['vma'])

implausible_vma = lieux_silver['vma'].notna() & (lieux_silver['vma'] > 130)
record('null_implausible_vma', 'lieux', int(implausible_vma.sum()), 'vma > 130 km/h treated as a keying error')
lieux_silver.loc[implausible_vma, 'vma'] = pd.NA

lieux_silver.to_parquet(SILVER / 'lieux.parquet', index=False)
print('dropped:', n0 - len(lieux_silver), '| implausible vma nulled:', int(implausible_vma.sum()))


dropped: 15846 | implausible vma nulled: 20


The vma count above is 20, not the 29 seen in Part 1 profiling. Worth tracing
rather than shrugging off: the dedup step just above runs first and removes 9 of those
29 rows along with the other 15,846 duplicates, because they belonged to accidents
whose first-listed `lieux` row wasn't the one carrying the bad `vma`. Re-running the
Part 1 count on the post-dedup frame confirms it, 29 drops to exactly 20.


In [23]:
vehicules_silver = load_raw('vehicules-2024.csv')
n0 = len(vehicules_silver)

vehicules_silver['catv'] = to_int_with_sentinel(vehicules_silver['catv'])  # raw CSV stores "7", not "07"
record('standardize_catv_zero_padding', 'vehicules', n0, 'cast catv to int so "7" and "07" are the same value everywhere downstream')

for col in ['senc', 'obs', 'obsm', 'choc', 'manv', 'motor']:
    vehicules_silver[col] = to_int_with_sentinel(vehicules_silver[col])
vehicules_silver['occutc'] = pd.to_numeric(vehicules_silver['occutc'].replace('', pd.NA), errors='coerce').astype('Int64')

vehicules_silver.to_parquet(SILVER / 'vehicules.parquet', index=False)


In [ ]:
usagers_silver = load_raw('usagers-2024.csv')
n0 = len(usagers_silver)

usagers_silver['grav'] = usagers_silver['grav'].astype(int)  # 0% missing, safe direct cast
usagers_silver['catu'] = usagers_silver['catu'].astype(int)
usagers_silver['sexe'] = to_int_with_sentinel(usagers_silver['sexe'])
usagers_silver['an_nais'] = pd.to_numeric(usagers_silver['an_nais'].replace('', pd.NA), errors='coerce').astype('Int64')
usagers_silver['age'] = 2024 - usagers_silver['an_nais']

# hit-and-run placeholder pattern from Part 1 B: flag it, don't impute a sex or age for someone who was never identified
fled_mask = usagers_silver['sexe'].isna() & (usagers_silver['grav'] == 1)
usagers_silver['fled_unidentified'] = fled_mask
record('flag_hit_and_run_placeholders', 'usagers', int(fled_mask.sum()), 'sexe/an_nais left null and flagged, not imputed')

for col in ['trajet', 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'etatp']:
    usagers_silver[col] = to_int_with_sentinel(usagers_silver[col])

severity_weight = {1: 0, 4: 1, 3: 3, 2: 6}
usagers_silver['severity_weight'] = usagers_silver['grav'].map(severity_weight)

non_ped = usagers_silver['catu'] != 3
seat_conflict = usagers_silver.duplicated(['Num_Acc', 'id_vehicule', 'place'], keep=False) & non_ped
usagers_silver['seat_conflict'] = seat_conflict
record('flag_seat_conflicts', 'usagers', int(seat_conflict.sum()), 'ambiguous (Num_Acc, id_vehicule, place) for non-pedestrians, flagged not deduped')

severity_rank_map = {2: 3, 3: 2, 4: 1, 1: 0}
usagers_silver['severity_rank'] = usagers_silver['grav'].map(severity_rank_map)

usagers_silver.to_parquet(SILVER / 'usagers.parquet', index=False)


Seat conflicts (1,459 rows, 1.26% of non-pedestrians) get flagged, not
deduplicated, they're different people sharing an ambiguous seat code, and collapsing
them would delete real injured or uninjured people from person-count and severity
totals. `severity_weight` (0/1/3/6 for indemne/léger/hospitalisé/tué) is deliberately
not evenly spaced, a fatality shouldn't be outvoted by 3 light-injury accidents in
a ranked list. That's a value judgment stated outright rather than hidden behind a
neutral-looking formula, in the spirit of KABCO-style weighting used elsewhere in
road-safety analytics, but picked by feel here, not lifted from a cited standard.
Anyone using this for an actual policy decision should swap in ONISR's own labelled
severity weighting instead.

Last piece: accident-level severity index and time-of-day, both derived from
`usagers_silver` and merged back onto `caracteristiques`.


In [25]:
acc_severity = usagers_silver.groupby('Num_Acc').agg(
    severity_index=('severity_weight', 'sum'),
    n_people=('id_usager', 'count'),
    n_killed=('grav', lambda s: (s == 2).sum()),
).reset_index()
worst_rank = usagers_silver.groupby('Num_Acc')['severity_rank'].max().rename('worst_severity_rank')
rank_to_label = {3: 'mortel', 2: 'hospitalise', 1: 'blesse_leger', 0: 'indemne'}
acc_severity = acc_severity.merge(worst_rank, on='Num_Acc')
acc_severity['accident_severity_class'] = acc_severity['worst_severity_rank'].map(rank_to_label)

def time_of_day(h):
    if h < 6: return 'night'
    if h < 9.5: return 'morning_rush'
    if h < 16: return 'midday'
    if h < 19.5: return 'evening_rush'
    return 'evening'

accidents['heure_float'] = accidents['datetime'].dt.hour + accidents['datetime'].dt.minute / 60
accidents['time_of_day'] = accidents['heure_float'].apply(time_of_day)
accidents = accidents.drop(columns='heure_float')
accidents = accidents.merge(acc_severity, on='Num_Acc', how='left')
record('enrich_severity_index_and_time_of_day', 'caracteristiques', n0, 'severity_index/accident_severity_class from usagers, time_of_day from hrmn')

accidents.to_parquet(SILVER / 'caracteristiques.parquet', index=False)

log_df = pd.DataFrame(transform_log)
log_df.to_csv(SILVER / 'transform_log.csv', index=False)
log_df


,step,table,rows_affected,note
0,standardize_datetime,caracteristiques,54402,combined jour/mois/an/hrmn into one datetime c...
1,standardize_coordinates,caracteristiques,54402,comma-decimal string to float (WGS84)
2,fix_coordinate_swap,caracteristiques,7,"lat/long swapped, rows outside their territory..."
3,null_unresolvable_coordinates,caracteristiques,3,"outside bbox and swap does not fix it, nulled ..."
4,dedup_lieux_to_one_per_accident,lieux,15846,"kept first row per Num_Acc, multi-road interse..."
5,null_implausible_vma,lieux,20,vma > 130 km/h treated as a keying error
6,standardize_catv_zero_padding,vehicules,92678,"cast catv to int so ""7"" and ""07"" are the same ..."
7,flag_hit_and_run_placeholders,usagers,2395,"sexe/an_nais left null and flagged, not imputed"
8,flag_seat_conflicts,usagers,1459,"ambiguous (Num_Acc, id_vehicule, place) for no..."
9,enrich_severity_index_and_time_of_day,caracteristiques,125187,severity_index/accident_severity_class from us...


`accident_severity_class` is built from an explicit rank (tué=3 > hospitalisé=2 >
léger=1 > indemne=0), not `min(grav)`, because the raw codes aren't severity-ordered
(2=tué is numerically small but the worst outcome), `min()` would silently mislabel
fatal accidents. Confirmed in the log above: 0 accidents classify as `indemne`,
matching the BAAC's own definition of "accident corporel" (requires at least one
victim), which is a nice independent check that the aggregation did the right thing.

`time_of_day` buckets (night/morning_rush/midday/evening_rush/evening) follow typical
French commute windows, not a fit to where this data's own hour-of-day distribution
actually inflects. That's the weakest justification in this notebook: a data-driven
version would be more defensible, skipped here to avoid tuning boundaries to one year's
noise, and I'd swap to a fitted version without much resistance if asked.

`data/silver/transform_log.csv` above is the machine-readable version of everything in
this section, one row per step with the exact count it touched. Re-running this
notebook on a new year's extract regenerates both the parquet files and this log
together, so they can't drift out of sync the way a hand-maintained changelog would.


In [26]:
print('Silver row counts:', {
    'caracteristiques': len(accidents), 'lieux': len(lieux_silver),
    'vehicules': len(vehicules_silver), 'usagers': len(usagers_silver),
})


Silver row counts: {'caracteristiques': 54402, 'lieux': 54402, 'vehicules': 92678, 'usagers': 125187}


### 2B. Gold layer modeling (star schema)

`fact_accidents` grain: one row per person involved in an accident (`Num_Acc` +
`id_usager`), not one row per accident. The obvious reading of "fact table (accidents)
+ dimensions as location, vehicle etc." is accident-grain, and I went with person-grain
instead for two reasons. `dim_vehicle` doesn't join cleanly at accident grain: an
accident can have several vehicles, and there's no defensible "primary vehicle" the way
there's a defensible "primary location" (2A above), so forcing accident-grain either
fans out the join anyway or picks one vehicle arbitrarily and loses the rest. `grav`,
the actual safety metric, is also measured per person, not per accident, a single
accident can have a hospitalised driver next to an unharmed passenger, which
accident-grain would flatten into one number before anyone could query them
separately. Person-grain is also a strict superset: every accident-level question is
still answerable by aggregating up (`nunique(location_id)`, `max(severity_rank)` per
accident), and the reverse isn't true. The cost, stated plainly: `COUNT(*)` on
`fact_accidents` counts people, not accidents, every "number of accidents" KPI in the
dashboard has to use `nunique()` on `location_id` instead, never a raw row count.

Reads the Silver parquet just written above, not the in-memory frames, so this section
stands on its own if re-run later.


In [27]:
GOLD = Path('data/gold')
GOLD.mkdir(exist_ok=True)

accidents_g = pd.read_parquet(SILVER / 'caracteristiques.parquet')
lieux_g = pd.read_parquet(SILVER / 'lieux.parquet')
vehicules_g = pd.read_parquet(SILVER / 'vehicules.parquet')
usagers_g = pd.read_parquet(SILVER / 'usagers.parquet')


`dim_location` (54,402 rows): one row per accident's recorded location, not
deduplicated across accidents that share a physical spot. Address text plus rounded GPS
can't reliably confirm "same real-world location", and building something that could
(geocoding, clustering) is a bigger effort than this exercise calls for.


In [28]:
dim_location = accidents_g[['Num_Acc', 'dep', 'com', 'agg', 'adr', 'lat', 'long']].merge(
    lieux_g[['Num_Acc', 'catr', 'voie', 'circ', 'nbv', 'vosp', 'prof', 'plan', 'larrout', 'surf', 'infra', 'situ', 'vma']],
    on='Num_Acc', how='left',
).rename(columns={'Num_Acc': 'location_id'})
dim_location.to_parquet(GOLD / 'dim_location.parquet', index=False)


In [ ]:
# junk dimension: one row per distinct (lum, atm, col) combo observed, not one per accident
cond_cols = ['lum', 'atm', 'col']
dim_conditions = accidents_g[cond_cols].drop_duplicates().reset_index(drop=True)
dim_conditions['condition_id'] = dim_conditions.index + 1
accidents_g = accidents_g.merge(dim_conditions, on=cond_cols, how='left')
dim_conditions.to_parquet(GOLD / 'dim_conditions.parquet', index=False)


In [30]:
accidents_g['hour'] = accidents_g['datetime'].dt.hour
dim_time = accidents_g[['date', 'hour', 'time_of_day']].drop_duplicates().reset_index(drop=True)
dim_time['time_id'] = dim_time.index + 1
dim_time['year'] = dim_time['date'].dt.year
dim_time['month'] = dim_time['date'].dt.month
dim_time['day'] = dim_time['date'].dt.day
dim_time['weekday'] = dim_time['date'].dt.day_name()
dim_time['is_weekend'] = dim_time['date'].dt.weekday >= 5
dim_time.to_parquet(GOLD / 'dim_time.parquet', index=False)


In [31]:
dim_vehicle = vehicules_g[['id_vehicule', 'catv', 'obs', 'obsm', 'choc', 'manv', 'motor', 'occutc', 'senc']].copy()
dim_vehicle.to_parquet(GOLD / 'dim_vehicle.parquet', index=False)

dim_user = usagers_g[[
    'id_usager', 'catu', 'sexe', 'an_nais', 'age', 'trajet',
    'secu1', 'secu2', 'secu3', 'locp', 'actp', 'etatp',
    'fled_unidentified', 'seat_conflict',
]].copy()
dim_user.to_parquet(GOLD / 'dim_user.parquet', index=False)


`severity_index`/`n_people`/`n_killed`/`accident_severity_class` are accident-level
and repeat on every person-row of the same accident, so anything aggregating them
needs `COUNT(DISTINCT location_id)`, never `COUNT(*)`.


In [32]:
time_lookup = accidents_g[['Num_Acc', 'date', 'hour', 'time_of_day']].merge(
    dim_time, on=['date', 'hour', 'time_of_day'], how='left'
)[['Num_Acc', 'time_id']]

fact_accidents = usagers_g[[
    'Num_Acc', 'id_usager', 'id_vehicule', 'grav', 'severity_weight', 'severity_rank',
]].merge(
    accidents_g[['Num_Acc', 'condition_id', 'severity_index', 'n_people', 'n_killed', 'accident_severity_class']],
    on='Num_Acc', how='left',
).merge(
    time_lookup, on='Num_Acc', how='left',
).rename(columns={'Num_Acc': 'location_id'})

fact_accidents.to_parquet(GOLD / 'fact_accidents.parquet', index=False)

print('Gold row counts:')
for name, df in [('fact_accidents', fact_accidents), ('dim_location', dim_location),
                  ('dim_conditions', dim_conditions), ('dim_time', dim_time),
                  ('dim_vehicle', dim_vehicle), ('dim_user', dim_user)]:
    print(f'  {name:16s} {len(df):>8,} rows, {df.shape[1]} columns')

print()
print('COUNT(*) vs COUNT(DISTINCT location_id):', len(fact_accidents), 'vs', fact_accidents['location_id'].nunique())


Gold row counts:
  fact_accidents    125,187 rows, 12 columns
  dim_location       54,402 rows, 19 columns
  dim_conditions        266 rows, 4 columns
  dim_time            8,961 rows, 9 columns
  dim_vehicle        92,678 rows, 9 columns
  dim_user          125,187 rows, 14 columns

COUNT(*) vs COUNT(DISTINCT location_id): 125187 vs 54402


Column reference, since the print above only gives shapes:

`fact_accidents`: `location_id` (FK → `dim_location`, = `Num_Acc`), `id_usager` (FK →
`dim_user`), `id_vehicule` (FK → `dim_vehicle`, the vehicle occupied or the one that
struck them if a pedestrian), `time_id` (FK → `dim_time`), `condition_id` (FK →
`dim_conditions`), `grav` (raw code, kept for codebook traceability), `severity_weight`,
`severity_rank`, `severity_index`, `n_people`, `n_killed`, `accident_severity_class`.

`dim_conditions` (266 rows): light/weather/collision-type each take a handful of
values, so the observed cross-product is small relative to 54,402 accidents, a proper
dimension instead of repeating three low-cardinality codes on every fact row.
`dim_time` (8,961 rows): one row per distinct `(date, hour, time_of_day)`, supports
both calendar-trend and time-of-day queries off the same table. `dim_vehicle` and
`dim_user` key off the already-clean natural keys `id_vehicule`/`id_usager`; `dim_user`
also carries the `fled_unidentified` and `seat_conflict` flags from Part 1.

What this supports: severity trend over time (`fact_accidents` → `dim_time`, group by
`month`, `nunique(location_id)` for accident counts, dedupe accident-level measures
before summing), spatial hotspots (join `dim_location`, weight points by
`severity_weight`), time-of-day pattern (group by `time_of_day`), and questions that
need person-grain to even ask, e.g. "are unbelted motorcyclists over-represented among
fatalities" (`dim_user` × `dim_vehicle`, filtered `grav=2`), unanswerable from an
accident-grain fact table.


### 2C. Medallion architecture diagram

Bronze (raw CSVs) → Silver (2A above) → Gold (2B above) → BI/dashboard.

![medallion architecture](medallion_architecture.png)

Row-count changes from Bronze to Silver are the point, not noise: `lieux` drops from
70,248 to 54,402 rows (the grain-violation dedup), the other 3 tables keep their
row counts and just gain cleaned, typed columns. Silver to Gold is a genuine grain
change, not a copy, `fact_accidents` is built at person-grain (125,187 rows) from
`usagers`, not accident-grain, for the reasons in 2B. The BI layer (`streamlit_app.py`)
only ever reads Gold, never Silver or Bronze directly, that boundary is what keeps the
dashboard simple: a handful of parquet reads, no cleaning logic duplicated in its code.


## Design justification

Five calls above weren't dictated by the data, collected here in one place since
that's how the brief lists it as a separate deliverable, even though each was already
argued next to the code that implements it.

**Fact table grain (2B).** Person, not accident. Costs one thing: `COUNT(*)` on
`fact_accidents` counts people, that caveat is repeated at every place in this notebook
that aggregates the fact table.

**`lieux` dedup rule (2A).** First row wins, arbitrary, and stated as such rather than
dressed up. Cheap to audit beats theoretically cleaner: a bridge table just relocates
the same pick to query time.

**Missing-GPS strategy.** There wasn't one to build, this extract has zero missing
coordinates. Assuming a problem exists and coding around it anyway would be the kind of
speculative work this project is trying to avoid.

**Severity weighting (2A).** 0/1/3/6, stretched on purpose, a value judgment stated
outright. Not a cited standard, replace with ONISR's own labelled weighting for
anything that isn't a class exercise.

**Time-of-day buckets (2A).** Commute intuition, not a fit to this data's actual
hour-of-day distribution. The weakest of the five, flagged as such rather than
presented with more confidence than it deserves.
